# Vector-Store Native Implementations (Chroma, Pinecone, Qdrant)

While frameworks like LangChain or LlamaIndex provide a unified abstract interface for metadata filtering, each vector database implements its own low-level query language, indexing strategy, and syntax payload rules.

When building production systems, knowing how different databases parse filters natively is critical for debugging performance bottlenecks.

## 1. Chroma (Local & Embedded Vector Store)
Chroma uses a dictionary-based syntax structure heavily inspired by MongoDB query operators. It supports logical operators like $and, $of, $eq, $ne, $gt, $gte, $lt, and $lte.

Native Chroma Filter Syntax

In [ ]:
# Chroma filter structure using standard dictionary operators
chroma_filter = {
    "$and": [
        {"department": {"$eq": "engineering"}},
        {"year": {"$gte": 2024}},
        {"confidential": {"$ne": True}}
    ]
}

results = vector_store.similarity_search(
    query="Architecture guidelines for microservices",
    k=4,
    filter=chroma_filter
)

Under the Hood: Chroma indexes metadata properties using SQLite (in local persistence mode) or direct in-memory dictionaries. Exact match lookups and range operations are evaluated instantly before passing remaining candidate IDs to the underlying hnswlib index.

## 2. Pinecone (Cloud-Native Managed Vector Database)
Pinecone handles metadata filtering via JSON objects using operators like $eq, $ne, $gt, $gte, $lt, $lte, $in, and $nin. It requires specific syntax structures depending on whether you are querying namespaces or metadata fields.

Native Pinecone Filter Syntax

In [ ]:
# Pinecone metadata filter syntax
pinecone_filter = {
    "genre": {"$in": ["sci-fi", "documentary"]},
    "year": {"$gte": 2023},
    "author": {"$eq": "Dr. Sarah Jenkins"}
}

# Passed directly to Pinecone index query
results = index.query(
    vector=[...], # query embedding vector
    top_k=5,
    include_metadata=True,
    filter=pinecone_filter
)

Under the Hood: Pinecone builds a metadata index alongside the vector index. Metadata filtering in Pinecone is tightly optimized for high throughput, making it exceptionally fast even with millions of vectors, provided the metadata fields are indexed.

## 3. Qdrant (High-Performance Vector Search Engine)
Qdrant uses a strongly-typed payload filter framework (models.Filter, models.FieldCondition, models.MatchValue, etc.). Rather than relying purely on loose dictionaries, Qdrant relies on explicit object declarations, making it robust against type mismatches.

Native Qdrant Filter Syntax (via Python Client)

In [ ]:
from qdrant_client.http import models

# Qdrant typed filter structure
qdrant_filter = models.Filter(
    must=[
        models.FieldCondition(
            key="department",
            match=models.MatchValue(value="engineering")
        ),
        models.FieldCondition(
            key="year",
            range=models.Range(gte=2024)
        )
    ]
)

results = vector_store.similarity_search(
    query="Cloud deployment setup",
    k=4,
    filter=qdrant_filter
)

Under the Hood: Qdrant maintains separate payload indexes (such as keyword, integer, or float indexes). When a filter is triggered, Qdrant calculates a bitmap of points matching the payload criteria and restricts the HNSW graph traversal to only those allowed points.

### Architectural Comparison Matrix for Native Filters


| Vector Store | Filter Paradigm | Key Operators Supported | Best Feature / Behavior
| :--- | :--- | :--- | :--- |
| Chroma | MongoDB-style dicts | "$eq, $ne, $gt, $in, $and, $or" | "Simple, lightweight, great for local prototyping and testing"
| Pinecone | JSON condition maps | "$eq, $ne, $gt, $gte, $in, $nin" | Highly scalable cloud-native filtering
| Qdrant | Strongly-typed objects | "models.Filter, MatchValue, Range" | "Advanced payload indexing, extreme speed on complex nested payloads"